# 01 — Data Preparation (9-class Nature Classifier)

Builds `train.csv`, `val.csv`, `test.csv` for the custom 9-class ResNet classifier.

**Classes:**
```
0 not_nature    — reject gate (urban/indoor scenes from Places365)
1 tree          — woody plants, large stature
2 shrub         — woody plants, smaller stature
3 grass_lawn    — grasses and lawn-forming species
4 mulch         — ⚠ sparse — realistic yield 500–1,200 images
5 garden_bed    — cultivated ornamental plantings
6 ground_cover  — low-growing spreading plants
7 green_roof    — vegetation on rooftops
8 water_body    — ponds, lakes, rivers, streams
```

**Datasets required — add via Kaggle → Add Data:**
- `noahbadoa/plantnet-300k-images` → classes 1–6 (taxonomy-based)
- `benjaminkz/places365` → classes 0, 3 (supplement), 5, 7, 8

**Target:** 5,000 images per class, 70 / 15 / 15 train/val/test split.  
Mulch and green_roof may fall short — use all available and document the gap.

**Expected runtime:** ~10 minutes (no GPU needed).

---
**Run cells in order. Read the audit outputs in cells 2 and 5 before proceeding.**

In [ ]:
import json
import os
import random
import pathlib
import collections

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

TARGET_PER_CLASS = 5_000
CLASSES = [
    "not_nature",   # 0
    "tree",         # 1
    "shrub",        # 2
    "grass_lawn",   # 3
    "mulch",        # 4
    "garden_bed",   # 5
    "ground_cover", # 6
    "green_roof",   # 7
    "water_body",   # 8
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# ── Locate datasets (3 levels deep) ───────────────────────────────────────────
def _find_dataset(keyword):
    root = pathlib.Path("/kaggle/input")
    for d1 in root.iterdir():
        if not d1.is_dir():
            continue
        if keyword in d1.name.lower():
            return d1
        for d2 in d1.iterdir():
            if not d2.is_dir():
                continue
            if keyword in d2.name.lower():
                return d2
            for d3 in d2.iterdir():
                if d3.is_dir() and keyword in d3.name.lower():
                    return d3
    return None

PLACES_ROOT   = _find_dataset("places")
PLANTNET_ROOT = _find_dataset("plantnet")

assert PLACES_ROOT is not None,   "Places365 not found"
assert PLANTNET_ROOT is not None, "PlantNet not found"

inner = PLANTNET_ROOT / "plantnet_300K"
if inner.exists():
    PLANTNET_ROOT = inner

print(f"Places365:  {PLACES_ROOT}")
print(f"PlantNet:   {PLANTNET_ROOT}")

# ── Places365 helpers ─────────────────────────────────────────────────────────
# Places365 ships train.txt: one line per image, format "rel/path/img.jpg class_id"
# Parsing this text file is ~5 seconds vs ~7 minutes of filesystem walking.

def _load_places_category_counts(places_root):
    """
    Parse train.txt to build {category_name: count} without reading any image
    file or directory listing.  Category = parent directory name of each image.
    Falls back to val.txt if train.txt is missing.
    """
    for fname in ("train.txt", "val.txt"):
        txt = pathlib.Path(places_root) / fname
        if not txt.exists():
            continue
        counts = collections.Counter()
        with open(txt) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rel = line.split()[0]           # "data_256/a/abbey/00001.jpg"
                cat = pathlib.Path(rel).parent.name  # "abbey"
                counts[cat] += 1
        return dict(counts), fname
    return {}, None

def _find_category_dir(places_root, category_name):
    """
    Locate the actual directory for a category.
    Tries the common Places365 layouts without walking the whole tree.
    """
    base_dirs = ["train", "val", "data_256", "data_large"]
    places_root = pathlib.Path(places_root)
    for sub in base_dirs:
        base = places_root / sub
        if not base.exists():
            continue
        # Layout A: places365/train/abbey/
        if (base / category_name).is_dir():
            return base / category_name
        # Layout B: places365/train/a/abbey/
        nested = base / category_name[0].lower() / category_name
        if nested.is_dir():
            return nested
    return None

IMAGE_EXTS = ('.jpg', '.jpeg', '.png')

def collect_from_places365(include=None, exclude=None, target=5000):
    """
    Collect image paths from Places365 by reading only the matched category
    directories — never walks the whole dataset.

    Args:
        include: keyword list — only keep categories whose name matches any
        exclude: keyword list — skip categories whose name matches any
        target:  max number of paths to return (0 = return nothing)
    """
    if target <= 0:
        return []

    matching = [
        cat for cat in PLACES_CATS
        if (not exclude or not any(kw in cat.lower() for kw in exclude))
        and (not include or any(kw in cat.lower() for kw in include))
    ]
    random.shuffle(matching)

    paths = []
    for cat in matching:
        cat_dir = _find_category_dir(PLACES_ROOT, cat)
        if cat_dir is None:
            continue
        imgs = [str(cat_dir / f) for f in os.listdir(cat_dir)
                if f.lower().endswith(IMAGE_EXTS)]
        paths.extend(imgs)
        if len(paths) >= target:
            break

    random.shuffle(paths)
    return paths[:target]

# ── Parse Places365 index ─────────────────────────────────────────────────────
print("\nParsing Places365 train.txt (sequential read, ~5 s)...")
PLACES_CATS, _src = _load_places_category_counts(PLACES_ROOT)
print(f"  {len(PLACES_CATS):,} categories loaded from {_src}")

# Sanity-check: confirm _find_category_dir resolves at least one category
_sample_cat = next(iter(PLACES_CATS))
_sample_dir = _find_category_dir(PLACES_ROOT, _sample_cat)
print(f"  Sample: '{_sample_cat}' → {_sample_dir}")

## Step 1 — Audit Places365 categories

Before writing any collection code, check what's actually available.
Places365-Standard caps at ~5,000 per category, but many relevant categories have far fewer.
Know the actual counts before planning.

In [ ]:
TARGET_KEYWORDS = [
    "roof", "garden", "mulch", "topiary", "lawn", "golf",
    "lake", "river", "pond", "ocean", "waterfall", "coast",
    "zen", "courtyard", "pasture", "athletic_field",
]

print(f"{'Category':<45} {'Images':>7}")
print("-" * 55)
for cat, count in sorted(PLACES_CATS.items()):
    if any(kw in cat.lower() for kw in TARGET_KEYWORDS):
        print(f"{cat:<45} {count:>7,}")

## Step 2 — `collect_from_categories` helper

In [ ]:
# collect_from_places365 is defined in cell 1 alongside the other helpers.
# This cell is kept as a placeholder to preserve the Step 2 section header above it.
print("collect_from_places365 ready.")

## Step 3 — PlantNet: taxonomy-based labelling

PlantNet organises images by numeric species ID. Random slicing would assign oak images to
the `grass_lawn` class. The taxonomy JSON maps each ID to a scientific name, and the genus
determines the category.

Run the genus audit print below before running collection. Review the output — bucket each
genus into a category or confirm it should be skipped. Most are immediately obvious.
Skip anything ambiguous (e.g. `prunus` covers both trees and shrubs).

In [ ]:
json_path = None
for candidate in pathlib.Path(_find_dataset("plantnet")).rglob("*.json"):
    if "species" in candidate.name.lower() and "name" in candidate.name.lower():
        json_path = candidate
        break

assert json_path is not None, "Species metadata JSON not found under PlantNet dataset"
print(f"Species JSON: {json_path}")

with open(json_path) as f:
    species_names = json.load(f)

# PlantNet JSON uses underscore-separated binomials: {"1234": "quercus_robur", ...}
# Genus = first component before the underscore.
all_genera = sorted(set(v.split('_')[0].lower() for v in species_names.values()
                        if isinstance(v, str)))
print(f"Unique genera in PlantNet-300K: {len(all_genera)}")
print()
print("All genera (review — decide tree/shrub/grass/garden/ground_cover/skip):")
for i in range(0, len(all_genera), 10):
    print("  ", "  ".join(f"{g:<18}" for g in all_genera[i:i+10]))

In [ ]:
CATEGORY_GENERA = {
    "tree": {
        "quercus", "fagus", "betula", "acer", "fraxinus", "tilia", "ulmus",
        "pinus", "abies", "picea", "larix", "cedrus", "taxus", "populus",
        "alnus", "carpinus", "platanus", "robinia", "castanea", "sorbus",
        "liriodendron", "metasequoia", "nothofagus",
        "tsuga", "thuja", "pseudotsuga", "liquidambar", "gleditsia",
        "catalpa", "gymnocladus", "celtis",
    },
    "shrub": {
        "rosa", "berberis", "cornus", "viburnum", "buddleja", "ligustrum",
        "lonicera", "sambucus", "cotoneaster", "crataegus", "ilex", "buxus",
        "lavandula", "salvia", "cytisus", "ulex", "calluna", "erica",
        "daphne", "nandina", "pieris",
        "vaccinium", "rubus", "amelanchier", "physocarpus", "spiraea",
        "syringa", "hydrangea",
    },
    "ground_cover": {
        "hedera", "vinca", "ajuga", "pachysandra", "hypericum", "sedum",
        "thymus", "fragaria", "potentilla", "trifolium", "lamium",
    },
    "grass_lawn": {
        "festuca", "poa", "lolium", "agrostis", "dactylis", "holcus",
        "bromus", "brachypodium", "arrhenatherum", "phleum",
    },
}

def assign_category(species_name):
    genus = species_name.split('_')[0].lower()
    for category, genera in CATEGORY_GENERA.items():
        if genus in genera:
            return category
    return None


images_train_dir = PLANTNET_ROOT / "images_train"
assert images_train_dir.exists(), f"images_train not found at {images_train_dir}"

categorised = collections.defaultdict(list)
skipped = 0

for species_id, species_name in species_names.items():
    if not isinstance(species_name, str):
        skipped += 1
        continue
    category = assign_category(species_name)
    if category is None:
        skipped += 1
        continue
    species_dir = images_train_dir / str(species_id)
    if species_dir.exists():
        imgs = list(species_dir.glob("*.jpg"))
        categorised[category].extend([str(p) for p in imgs])

print(f"Species skipped (ambiguous/unmapped): {skipped}")
print()
print("PlantNet images per class (before supplement):")
for cls in ["tree", "shrub", "grass_lawn", "ground_cover"]:
    n = len(categorised.get(cls, []))
    status = "✓" if n >= TARGET_PER_CLASS else f"⚠  only {n:,}"
    print(f"  {cls:<15}: {n:7,}  {status}")

# Supplement sparse tree class with Places365 tree_farm.
# PlantNet-300K lacks most common temperate tree genera; tree_farm gives
# row-planted trees which are valid training signal for trunk/canopy recognition.
_tree_gap = max(0, TARGET_PER_CLASS - len(categorised["tree"]))
if _tree_gap > 0:
    _tree_supp = collect_from_places365(include=["tree_farm"], target=_tree_gap)
    categorised["tree"].extend(_tree_supp)
    print(f"\n  tree after tree_farm supplement: {len(categorised['tree']):,}")

## Step 4 — `not_nature` (Places365)

The `not_nature` class does almost all the safety-critical work. Clean indoor/urban images
are necessary but not sufficient — the real failure modes are things that *look like nature*
but aren't qualifying submissions:

- Potted plants / houseplants (real plants, not a nature element in the arboretum)
- Garden centre shelves (plants in plastic pots for sale)
- Artificial / synthetic turf
- Nature paintings or photos on walls

If Places365 doesn't cover these, manually curate 200–300 images each and add as a
supplementary Kaggle dataset. **200 hard negatives will do more than 5,000 more office photos.**

Audit the matched category list printed below before using it.

In [ ]:
NATURE_KEYWORDS = [
    # vegetation / landscape
    "forest", "park", "garden", "beach", "mountain", "field", "meadow",
    "lake", "river", "ocean", "coast", "cliff", "valley", "jungle", "swamp",
    "waterfall", "glacier", "desert", "woodland", "nature", "grass", "lawn",
    "greenhouse", "nursery", "wetland", "bog", "pond", "pasture",
    # missed in previous version — revealed by audit output
    "marsh", "tundra", "creek", "lagoon", "canal", "orchard", "vineyard",
    "golf",         # golf_course belongs in grass_lawn, not not_nature
    "campsite",     # outdoor with vegetation
    "tree_farm",    # trees
    "rice_paddy",   # agricultural vegetation
    "moat",         # water
    "swimming_hole", "hot_spring", "watering_hole",  # water bodies
]

matched_not_nature = sorted(
    cat for cat in PLACES_CATS
    if not any(kw in cat.lower() for kw in NATURE_KEYWORDS)
)
print(f"Categories included in not_nature ({len(matched_not_nature)} total):")
for name in matched_not_nature:
    print(f"  {name}")
print()
print("Scan this list. Remove any that look nature-adjacent before proceeding.")

In [ ]:
not_nature_paths = collect_from_places365(
    exclude=NATURE_KEYWORDS,
    target=TARGET_PER_CLASS,
)
print(f"not_nature: {len(not_nature_paths):,} images")

## Step 5 — Remaining classes from Places365

In [ ]:
water_paths = collect_from_places365(
    include=["lake", "river", "pond", "ocean", "coast", "reservoir", "canal",
             "waterfall", "fishpond", "creek", "lagoon", "marsh", "moat",
             "swimming_hole", "hot_spring", "watering_hole", "wave",
             "underwater"],
    target=TARGET_PER_CLASS,
)
print(f"water_body:   {len(water_paths):,}")

grass_shortfall = max(0, TARGET_PER_CLASS - len(categorised["grass_lawn"]))
grass_supplement = collect_from_places365(
    include=["lawn", "golf_course", "pasture", "athletic_field"],
    target=grass_shortfall,
)
print(f"grass_lawn supplement: {len(grass_supplement):,} (shortfall was {grass_shortfall:,})")

garden_paths = collect_from_places365(
    include=["formal_garden", "botanical_garden", "flower_garden", "rose_garden",
             "japanese_garden", "vegetable_garden"],
    target=TARGET_PER_CLASS,
)
print(f"garden_bed:   {len(garden_paths):,}")

green_roof_paths = collect_from_places365(
    include=["roof_garden", "rooftop"],
    target=TARGET_PER_CLASS,
)
print(f"green_roof:   {len(green_roof_paths):,}  (target {TARGET_PER_CLASS:,})")

mulch_paths = collect_from_places365(
    include=["topiary_garden", "zen_garden", "courtyard"],
    target=TARGET_PER_CLASS,
)
print(f"mulch:        {len(mulch_paths):,}  (target {TARGET_PER_CLASS:,}) — shortfall expected")

## Step 6 — Build DataFrame and stratified split

In [ ]:
all_sources = {
    "not_nature":   not_nature_paths,
    "tree":         categorised["tree"],
    "shrub":        categorised["shrub"],
    "grass_lawn":   categorised["grass_lawn"] + grass_supplement,
    "mulch":        mulch_paths,
    "garden_bed":   garden_paths,
    "ground_cover": categorised["ground_cover"],
    "green_roof":   green_roof_paths,
    "water_body":   water_paths,
}

rows = []
for class_name, paths in all_sources.items():
    random.shuffle(paths)
    sample = paths[:TARGET_PER_CLASS]
    label = CLASS_TO_IDX[class_name]
    for p in sample:
        rows.append((p, class_name, label))

df = pd.DataFrame(rows, columns=["path", "class_name", "label"])
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Stratified split preserves class proportions in every split.
# With balanced classes this has little practical effect, but it's free insurance
# when mulch/green_roof are significantly under TARGET_PER_CLASS.
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv",     index=False)
test_df.to_csv("/kaggle/working/test.csv",   index=False)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print()
# Expected: each class ~3,500 in train. Mulch/green_roof may be lower — document this.
print(train_df["class_name"].value_counts().reindex(CLASSES).to_string())

## Step 7 — Visual QA

Display a 3×3 grid of random images for each class.
This is the last chance to catch labelling errors before training.

Specifically check:
- **tree**: canopy/trunk visible, not just bushes
- **ground_cover**: low-growing plants, not grass patches
- **garden_bed**: planted soil/flower beds, not random outdoor scenes
- **mulch**: wood chips/bark coverage — check for hedges and gravel (topiary/zen_garden contamination)
- **green_roof**: vegetation ON a rooftop, not a regular rooftop
- **not_nature**: no potted plants, no garden centres, no artificial turf —
  if these are present, add hard negatives before training

In [ ]:
N_COLS = 3
present = [c for c in CLASSES if c in train_df["class_name"].values]

fig, axes = plt.subplots(
    len(present), N_COLS,
    figsize=(N_COLS * 2.5, len(present) * 2.5)
)
if len(present) == 1:
    axes = [axes]

for row_axes, cls in zip(axes, present):
    subset = train_df[train_df["class_name"] == cls]
    sample_paths = subset.sample(
        min(N_COLS, len(subset)), random_state=SEED
    )["path"].tolist()

    for ax, path in zip(row_axes, sample_paths):
        try:
            ax.imshow(Image.open(path).convert("RGB").resize((128, 128)))
        except Exception:
            ax.text(0.5, 0.5, "Error", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

    row_axes[0].set_ylabel(cls, rotation=0, labelpad=80, va="center", fontsize=9)

    # Blank any unused columns if this class had fewer than N_COLS images
    for ax in row_axes[len(sample_paths):]:
        ax.axis("off")

plt.suptitle("3×3 visual QA — review for labelling errors", y=1.01)
plt.tight_layout()
plt.show()

## Step 8 — Class balance summary

In [ ]:
counts = train_df["class_name"].value_counts().reindex(CLASSES).fillna(0)

fig, ax = plt.subplots(figsize=(11, 4))
colors = [
    "#ef4444" if v < TARGET_PER_CLASS * 0.70 * 0.70  # below 70% of train target
    else "#f59e0b" if v < TARGET_PER_CLASS * 0.70      # below full train target
    else "#22c55e"
    for v in counts.values
]
ax.bar(counts.index, counts.values, color=colors)
ax.axhline(TARGET_PER_CLASS * 0.70, color="gray", linestyle="--", label="Full target (train)")
ax.set_title("Training set class balance")
ax.set_ylabel("Images")
ax.legend()
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print("\nFull dataset per-class counts:")
print(df["class_name"].value_counts().reindex(CLASSES).to_string())
print("\nCSV files saved to /kaggle/working/")